# Session 1: When Optimal Fails — Stress-Testing Classical Minimum-Variance Portfolios

Modern portfolio theory promises an "optimal" allocation — but optimal under what assumptions? In this session, we'll build a classical minimum-variance portfolio and then systematically break it. We'll use AI-driven "what-if" stress tests to expose how fragile textbook solutions can be when the world doesn't cooperate.

> __Learning Objectives:__
>
> * **Minimum-Variance Portfolio Construction**: Formulate and solve the classical Markowitz quadratic program to find the lowest-risk allocation for a given return target
> * **Weaknesses of Mean-Variance Optimization**: Identify and explain three systemic failure modes — input sensitivity, weight concentration, and assumption fragility — that cause "optimal" portfolios to underperform in practice
> * **AI-Powered Stress Testing**: Design and automate correlation-shift, price-drop, and trading-cost stress tests that systematically violate the optimizer's assumptions
> * **Baseline Performance Scorecard**: Compute a four-metric scorecard (expected return, maximum drawdown, turnover, and trading cost) that serves as the benchmark for AI-driven improvements in Sessions 2–4

## Examples
The following example notebooks accompany this lecture. They contain executable code that implements the concepts discussed here:

[▶ Let's build a classical minimum-variance portfolio from synthetic data](eCornell-AI-Finance-L1-Example-BuildMinVariancePortfolio-May-2026.ipynb). We generate a synthetic asset universe, estimate $\boldsymbol{\mu}$ and $\boldsymbol{\Sigma}$, solve the quadratic program, compute the efficient frontier, and explore input sensitivity.

[▶ Let's stress-test the minimum-variance portfolio](eCornell-AI-Finance-L1-Example-StressTestMinVariancePortfolio-May-2026.ipynb). We run systematic "what-if" experiments — shifting correlations, simulating price drops, increasing trading costs — and produce a baseline performance scorecard with Monte Carlo confidence bands.

## From Prices to Growth Rates
The inputs to portfolio optimization — expected returns and covariances — don't fall from the sky. They must be _estimated_ from observed price data. The starting point is the _log growth rate_, which converts a time series of asset prices into a stationary series of returns suitable for statistical analysis.

Suppose we observe the closing price $S_i(t_k)$ of asset $i$ at discrete times $t_0, t_1, \ldots, t_T$, spaced $\Delta t$ apart (typically $\Delta t = 1/252$ for daily data). The log growth rate of asset $i$ between times $t_{k-1}$ and $t_k$ is:

$$\boxed{g_i(t_k) = \frac{1}{\Delta t}\ln\frac{S_i(t_k)}{S_i(t_{k-1})}}$$

> __Why log returns?__ Log returns have two important properties: (1) they are additive over time ($g$ over two days equals the sum of the daily $g$'s), and (2) for small returns, $g \approx (S_{t} - S_{t-1})/S_{t-1}$, i.e., the log return is close to the simple percentage return. Annualizing by dividing by $\Delta t$ puts growth rates on a consistent scale regardless of the observation frequency.

Given $T$ observations for each of $N$ assets, we assemble the growth rates into a _data matrix_ $\mathbf{X} \in \mathbb{R}^{(T-1) \times N}$ where each row is a time step and each column is an asset. This matrix is the raw material for estimating the inputs to the optimizer.

## Portfolio Reward and Portfolio Risk
A portfolio $\mathcal{P}$ is defined by a weight vector $\mathbf{w} = (w_1, \ldots, w_N)^{\top}$ where $w_i$ is the fraction of total wealth invested in asset $i$. The portfolio's growth rate at time $t_k$ is the weighted sum of the individual asset growth rates:

$$g_{\mathcal{P}}(t_k) = \sum_{i=1}^{N} w_i\, g_i(t_k) = \mathbf{w}^{\top}\mathbf{g}(t_k)$$

### Portfolio Reward
The _expected_ portfolio growth rate is the weighted average of the individual expected growth rates:

$$\boxed{\mathbb{E}\left[g_{\mathcal{P}}\right] = \sum_{i=1}^{N} w_i\,\mathbb{E}\left[g_i\right] = \mathbf{w}^{\top}\boldsymbol{\mu}}$$

where $\boldsymbol{\mu} = \bigl(\mathbb{E}[g_1],\ldots,\mathbb{E}[g_N]\bigr)^{\top}$ is the expected growth rate vector. This is the portfolio's _reward_ — the return we expect to earn.

### Portfolio Risk
The _variance_ of the portfolio growth rate measures how much the portfolio's return fluctuates around its expectation:

$$\boxed{\text{Var}\left(g_{\mathcal{P}}\right) = \mathbf{w}^{\top}\boldsymbol{\Sigma}\,\mathbf{w}}$$

where $\boldsymbol{\Sigma} \in \mathbb{R}^{N \times N}$ is the covariance matrix of asset growth rates. The $(i,j)$ entry of $\boldsymbol{\Sigma}$ is $\sigma_{ij} = \text{Cov}(g_i, g_j)$, and the diagonal entry $\sigma_{ii} = \text{Var}(g_i) = \sigma_i^2$.

> __Decomposing $\boldsymbol{\Sigma}$.__ The covariance matrix can be decomposed as $\boldsymbol{\Sigma} = \mathbf{D}\,\mathbf{C}\,\mathbf{D}$ where $\mathbf{D} = \text{diag}(\sigma_1,\ldots,\sigma_N)$ contains the asset volatilities and $\mathbf{C}$ is the _correlation matrix_ with $C_{ij} = \sigma_{ij}/(\sigma_i\sigma_j)$. This decomposition is useful because volatilities and correlations have different estimation properties and behave differently under stress.

> __Growth Rate versus Simple Return.__ We work with log growth rates $g_i$ rather than simple returns $r_i = (S_{t} - S_{t-1})/S_{t-1}$ because log returns are additive over time and lead to cleaner statistical properties. For small returns, $g_i \approx r_i$, so the distinction matters most for assets with high volatility or long holding periods.

## Estimating $\boldsymbol{\mu}$ and $\boldsymbol{\Sigma}$ from Data
In practice, we never know the true expected growth rates or the true covariance structure — we estimate them from a historical sample of $T$ observations. Given the data matrix $\mathbf{X} \in \mathbb{R}^{T \times N}$, the standard estimators are:

**Sample mean** (expected growth rate vector):

$$\hat{\boldsymbol{\mu}} = \frac{1}{T}\sum_{t=1}^{T}\mathbf{x}_t$$

**Sample covariance matrix**:

$$\boxed{\hat{\boldsymbol{\Sigma}} = \frac{1}{T-1}\sum_{t=1}^{T}\left(\mathbf{x}_t - \hat{\boldsymbol{\mu}}\right)\left(\mathbf{x}_t - \hat{\boldsymbol{\mu}}\right)^{\top} = \frac{1}{T-1}\tilde{\mathbf{X}}^{\top}\tilde{\mathbf{X}}}$$

where $\tilde{\mathbf{X}}$ is the _centered_ data matrix with the sample mean subtracted from each row.

> __The estimation asymmetry.__ Expected returns are notoriously hard to estimate because the _signal_ (the mean growth rate, typically a few basis points per day) is tiny relative to the _noise_ (daily volatility, typically 50–150 bps). With 2 years of daily data ($T \approx 504$), the standard error of the mean is $\sigma/\sqrt{T} \approx \sigma/22$, which is still large relative to $\mu$. The covariance matrix is estimated much more precisely because it measures the _spread_ of the data, not its center.
>
> This asymmetry is the root cause of the "error maximization" problem: the optimizer's most important input ($\boldsymbol{\mu}$) is its least reliable one.

> __Alternative estimators.__ Practitioners often replace the sample covariance with _shrinkage estimators_ (e.g., Ledoit-Wolf) or _factor models_ (e.g., the single-index model we'll use in Session 2) that trade some bias for lower estimation variance. For expected returns, Black-Litterman and other Bayesian approaches combine market equilibrium with investor views. These are active research areas — for now, we use the sample estimators to establish the baseline.

## The Classical Minimum-Variance Portfolio
In 1952, [Harry Markowitz](https://doi.org/10.2307/2975974) proposed a deceptively simple idea: investors care about the _expected return_ and the _variance_ (risk) of their portfolio, and they want the best trade-off between the two. The minimum-variance portfolio is the allocation that achieves the lowest possible risk for a given level of expected return.

With the estimated expected growth rate vector $\hat{\boldsymbol{\mu}}$ and covariance matrix $\hat{\boldsymbol{\Sigma}}$ from the previous section in hand, we can now formulate the optimization problem. We seek a weight vector $\mathbf{w}\in\mathbb{R}^{N}$ that solves the quadratic program:

$$\boxed{\begin{align*}\min_{\mathbf{w}} &\quad \mathbf{w}^{\top}\hat{\boldsymbol{\Sigma}}\,\mathbf{w} \\ \text{subject to:} &\quad \mathbf{w}^{\top}\hat{\boldsymbol{\mu}} \geq R \\ &\quad \sum_{i=1}^{N} w_{i} = 1 \\ &\quad l_{i} \leq w_{i} \leq u_{i} \quad \forall\, i\end{align*}}$$

where $R$ is the target return and $l_i, u_i$ are the lower and upper bounds on each weight (e.g., no short selling means $l_i = 0$).

> __Interpretation.__ The objective $\mathbf{w}^{\top}\hat{\boldsymbol{\Sigma}}\,\mathbf{w}$ is the portfolio variance — the reward formula from the previous section. The first constraint ensures we achieve at least a target return $R$. The budget constraint forces the weights to sum to 1 (fully invested). The box constraints enforce position limits.

This is a convex quadratic program — it has a unique global minimum and can be solved efficiently using solvers such as [Ipopt](https://coin-or.github.io/Ipopt/) or [COSMO](https://github.com/oxfordcontrol/COSMO.jl). On the surface, it seems like the perfect recipe for rational investing. But there's a catch: _everything depends on the quality of $\hat{\boldsymbol{\mu}}$ and $\hat{\boldsymbol{\Sigma}}$_ — and as we discussed above, $\hat{\boldsymbol{\mu}}$ is estimated with far more error than $\hat{\boldsymbol{\Sigma}}$.

### The Efficient Frontier
By sweeping the target return $R$ from its minimum to its maximum feasible value and solving the quadratic program at each point, we trace out the _efficient frontier_ — the set of portfolios that achieve the lowest risk for each level of return.

<div>
    <center>
        <img src="figs/Fig-MinVar-Portfolio-RA-Schematic.png" width="680"/>
    </center>
</div>

The figure above illustrates the key features of the risk-reward landscape:

- **Portfolio 1** (red, upper frontier) is an efficient portfolio with high expected reward $\mathbb{E}(R_1)$ but also high risk $\sigma_1$. It lies _on_ the efficient frontier.
- **Portfolio 2** (blue, min-var) is the _minimum-variance portfolio_ — it has the lowest achievable risk $\sigma_2$ among all feasible allocations. It anchors the left end of the frontier and earns expected reward $\mathbb{E}(R_2)$.
- **Portfolio 3** (open circle, below frontier) is _suboptimal_: it has the same risk $\sigma_1$ as Portfolio 1, but lower expected reward $\mathbb{E}(R_3)$. A rational investor should move _up_ to the frontier.
- The **solid red curve** is the efficient frontier — every point on it is Pareto optimal. The **dashed curve** below the min-var portfolio represents feasible but dominated allocations.

> __The investor's trade-off.__ Moving along the frontier from Portfolio 2 to Portfolio 1 increases both reward _and_ risk. The curvature of the frontier encodes the _cost of chasing return_ — initially, a small increase in risk buys a lot of extra return, but the marginal benefit diminishes as we move further right. The shape of this curve depends entirely on $\hat{\boldsymbol{\mu}}$ and $\hat{\boldsymbol{\Sigma}}$ — which is why estimation quality matters so much.

## When Optimal Fails: Three Weaknesses of Mean-Variance Optimization

The minimum-variance framework is elegant, but practitioners have learned — often painfully — that it has three systemic weaknesses:

### 1. Input Sensitivity (Error Maximization)
The optimizer treats $\boldsymbol{\mu}$ and $\boldsymbol{\Sigma}$ as _ground truth_. But these are estimated from historical data, and small estimation errors can produce wildly different optimal weights. [Michaud (1989)](https://doi.org/10.2469/faj.v45.n1.31) coined the term _"error maximizer"_ to describe mean-variance optimization: the optimizer aggressively exploits estimation errors, loading up on assets whose returns are _overestimated_ and avoiding those that are _underestimated_.

> __The Estimation Problem.__ Expected returns are notoriously difficult to estimate. A change of just 50 basis points in one asset's expected return can shift its optimal weight by 20% or more. The covariance matrix is more stable, but still subject to estimation error — especially off-diagonal terms (correlations) during periods of market stress. [Chopra and Ziemba (1993)](https://doi.org/10.3905/jpm.1993.409440) showed that errors in means are roughly 10× more damaging than errors in variances.

### 2. Weight Concentration
Without tight constraints, minimum-variance portfolios tend to concentrate in a small number of assets — often the ones with the lowest historical variance. This produces portfolios that _look_ optimal on paper but carry hidden concentration risk.

> __Why does this happen?__ The quadratic objective $\mathbf{w}^{\top}\boldsymbol{\Sigma}\mathbf{w}$ rewards low-variance assets disproportionately. If two assets have similar expected returns but one has slightly lower variance, the optimizer will heavily favor the lower-variance asset. In practice, this means a "diversified" portfolio can end up with 60–80% of its weight in just 2–3 names.

### 3. Assumption Fragility
The entire framework assumes that returns are drawn from a stationary multivariate distribution — that the future will look like the past. This assumption breaks during exactly the moments when portfolio construction matters most: regime changes, market crises, and structural shifts.

> __Stationarity is a luxury.__ Correlations spike during crises (assets that were "diversifying" suddenly move together), volatilities jump, and expected returns shift. The portfolio that was "optimal" under calm conditions can become the _worst_ possible allocation when the regime changes. This phenomenon — sometimes called _correlation breakdown_ — is one of the central motivations for the HMM-based approach we'll explore in Session 3.

## Stress Testing: Using AI to Break Your Own Portfolio

If the weakness of mean-variance optimization is its dependence on assumptions, the remedy is to _systematically violate those assumptions_ and see what happens. This is the idea behind stress testing.

We'll design three categories of "what-if" experiments:

### Test 1: Correlation Shifts
What happens to our portfolio if pairwise correlations increase by 10%, 25%, or 50%? During crises, correlations tend to spike toward 1.0 — the diversification benefit we counted on evaporates. We perturb the correlation matrix $\mathbf{C}$ as:

$$\boxed{\tilde{\mathbf{C}} = (1-\alpha)\,\mathbf{C} + \alpha\,\mathbf{1}\mathbf{1}^{\top}, \qquad \alpha\in[0,1]}$$

> __Interpretation.__ At $\alpha = 0$ we have the original correlations; at $\alpha = 1$, all assets are perfectly correlated. Intermediate values simulate partial correlation breakdown — the kind of regime we observe during market stress events.

### Test 2: Price Drops
What if one or more assets experience a sudden price decline? We simulate scenarios where individual assets or sectors drop by 10%, 20%, or 40%, and compute the portfolio's loss exposure. This tests the portfolio's _tail risk_ — the risk that concentration in a few assets amplifies downside shocks.

> __Why this matters.__ A portfolio with 60% weight in a single asset will lose 24% of its value if that asset drops 40%. An equal-weight portfolio of 5 assets would lose only 8%. Stress testing makes this concentration risk visible _before_ the drop happens.

### Test 3: Higher Trading Costs
What if transaction costs are 2×, 5×, or 10× our baseline estimate? High-turnover portfolios that look optimal under low-cost assumptions can become net losers when costs rise — for instance during periods of low liquidity or market stress.

> __The cost of rebalancing.__ Every time we trade to maintain "optimal" weights, we pay a cost. The turnover metric measures how much trading is required; the trading cost metric converts that into dollars. Together, they tell us whether the optimization's theoretical gains survive contact with the real market.

In each case, we re-solve the optimization under the stressed inputs and compare the resulting portfolio to the baseline. The AI's role here is to _automate the generation and evaluation_ of hundreds of stress scenarios, producing a comprehensive map of how the portfolio degrades.

## The Baseline Scorecard

Before we can evaluate any improvements (that's Sessions 2–4), we need a _baseline_ — a quantitative record of how the classical minimum-variance portfolio performs under both normal and stressed conditions. Our scorecard tracks four metrics:

| Metric | Definition | Why It Matters |
|:-------|:----------|:---------------|
| **Expected Return** | $\mathbf{w}^{\top}\boldsymbol{\mu}$ | Does the portfolio meet our return target? |
| **Maximum Drawdown** | Largest peak-to-trough decline in cumulative wealth | How bad does the worst loss get? |
| **Turnover** | $\sum_{i=1}^{N}\lvert w_{i,t} - w_{i,t-1}\rvert$ | How much trading is required to maintain the allocation? |
| **Trading Cost** | Turnover $\times$ cost-per-unit-traded | What is the real cost of rebalancing? |

> __Why these four?__ Return tells us if the strategy is worth pursuing. Drawdown tells us if we can survive the journey. Turnover tells us how active the strategy is. Trading cost tells us if the activity is economically justified. Together, they form a _necessary and sufficient_ baseline for comparing strategies.

The maximum drawdown deserves a formal definition. Given a cumulative wealth series $W_t$, define the running peak as $\hat{W}_t = \max_{s \leq t} W_s$. Then:

$$\boxed{\text{MaxDrawdown} = \max_{t}\;\frac{\hat{W}_t - W_t}{\hat{W}_t}}$$

> __Monte Carlo Confidence Bands.__ In the example notebooks, we won't just compute a single drawdown number — we'll simulate many return paths from the estimated distribution and compute the scorecard across all paths. This gives us a _distribution_ of outcomes (with 68%, 95%, and 99% confidence bands), not just a point estimate. This is the AI-powered analog of classical stress testing: instead of manually specifying scenarios, we let the model generate them.

We'll compute this scorecard under the baseline assumptions and under each stress scenario. The result is a matrix of outcomes that reveals exactly where the classical approach breaks down — and that sets the stage for Session 2, where we'll design an AI-powered rebalancing engine to address these weaknesses.

## Summary

In this session we established the classical minimum-variance portfolio as our starting point, examined why it fails in practice, and designed a stress-testing framework to quantify those failures.

> __Key Takeaways:__
>
> * **The Markowitz quadratic program** produces a unique, globally optimal allocation — but its solution quality is _entirely dependent_ on the accuracy of estimated returns $\boldsymbol{\mu}$ and covariances $\boldsymbol{\Sigma}$
> * **Three systemic weaknesses** — input sensitivity (error maximization), weight concentration, and assumption fragility — mean that "optimal" portfolios can fail precisely when they are needed most
> * **Systematic stress testing** (correlation shifts, price drops, cost increases) reveals the _boundary conditions_ under which the classical approach degrades, and Monte Carlo simulation quantifies the uncertainty around each metric
> * **The baseline scorecard** (return, drawdown, turnover, trading cost) gives us a quantitative benchmark that we'll use throughout the remaining sessions to evaluate AI-driven improvements

___

__What comes next?__ In [Session 2](../session-2/eCornell-AI-Finance-L2-Lecture-AIRebalancingEngine-May-2026.ipynb), we move from static weights to adaptive allocation — designing an AI rebalancing engine that uses Cobb-Douglas utility maximization with sentiment-driven preference weights to respond dynamically to changing market conditions.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The examples use synthetic data and simplified models. Real-world portfolio construction involves additional considerations including regulatory requirements, tax implications, liquidity constraints, and operational risks that are beyond the scope of this session.